# Writing Bake-off — Round 6: Nemotron 3 Ultra

Head-to-head against a config carried over from round 5. Same ten topics, two configs, **20 calls**.

## Data-path guard, read this first

The OpenRouter listing carries **two endpoints**. The free one logs your sessions to NVIDIA
for product improvement. Confidential material must not go there.

Every Nemotron call in this notebook sends:

```json
"provider": {
  "only": ["deepinfra", "baseten"],
  "data_collection": "deny",
  "allow_fallbacks": false
}
```

`data_collection: "deny"` restricts routing to providers that do not store data
non-transiently. `only` plus `allow_fallbacks: false` means the request fails loudly rather
than silently landing somewhere else. The run records which provider actually served each
call so you can verify after the fact rather than trust the config.

These three settings apply to the Nemotron route only. The Opus route has no provider restriction in this
notebook: its request carries only a price limit, so OpenRouter picks the provider. **Do not put
confidential or client material into prompts sent on the Opus route** unless you have checked that
provider's data policy yourself.

The run cell refuses to start unless both probe calls succeeded and the Nemotron probe named a provider on that
allowlist. A missing, empty or non-string provider name counts as a failure, and so does a probe that
failed, raised or was never run. Names are compared exactly after trimming spaces and lowercasing, never by
substring. The probe result is bound to the model ids and the allowlist it was taken with, so a result from
an earlier probe cannot be reused after either one changes.

DeepInfra lists at $0.50/$2.20 and Baseten at $0.60/$2.40; the notebook reserves spend at the highest
confirmed OpenRouter endpoint price ($0.625/$3.125) and flags any Nemotron call served elsewhere.

## Why this is worth a run

If you apply a country-of-origin policy to models, check it first. NVIDIA is a US company
and the weights are open. At roughly a tenth of the comparison model's rate, a tie would be
a real result.

## Reasons for a skeptical prior

Published Artificial Analysis figures put it at **22.6% accuracy on AA-Omniscience** with a
**70.3% non-hallucination rate**, and it is positioned for agentic orchestration and coding
rather than prose. Watch for a model built for agentic work reciting the voice card as
content instead of writing the post.

The test is cheap and the prediction is falsifiable. That is enough reason to run it.

## Design

Two configs, ten topics, one rep. Paired A/B per topic, letters reshuffled each time.
Straight win count; five of ten is chance, eight of ten is a real gap.

**Fill the placeholders first.** The briefs contain `<...>` placeholders and `[Placeholder ...]`
markers. Replace them with your own material before you run.

## 1. Setup

In [ ]:
import os, sys, subprocess, json, re, html, time, random
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    import httpx
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx"], check=True)
    import httpx

VOLUME  = Path("./local-data")
ROOT    = VOLUME / "bakeoff_r6_nemotron"
SOURCES = VOLUME / "writing_bakeoff" / "sources"
RAW     = ROOT / "raw_results.jsonl"
READING = ROOT / "READING_PACK.md"
ANSWERS = ROOT / "ANSWER_KEY.md"
RANKS   = ROOT / "rankings.csv"

ROOT.mkdir(parents=True, exist_ok=True)
SOURCES.mkdir(parents=True, exist_ok=True)

print("ready ->", ROOT if os.environ.get("OPENROUTER_API_KEY") else "SET OPENROUTER_API_KEY FIRST")

## 2. Config

`SAFE_PROVIDERS` is the whole point of this cell. Change it and you may route to the free
NVIDIA endpoint.

In [ ]:
# Providers permitted to serve Nemotron. The free NVIDIA endpoint is excluded by both the
# allow-list and the data_collection setting; belt and braces on purpose.
SAFE_PROVIDERS = ["deepinfra", "baseten"]


def _norm_provider(p):
    """Comparison form of a provider name: surrounding whitespace stripped, lowercased. Anything that is not a
    non-empty string has no name and gives None."""
    if isinstance(p, str) and p.strip():
        return p.strip().lower()
    return None


def provider_ok(cfg, provider):
    """True only if cfg has no provider allowlist, or `provider` is an explicit name that equals, after _norm_provider,
    an entry of the allowlist the request-level provider setting sends (MODELS[cfg]["providers"]). Exact match only."""
    allowed = MODELS[cfg]["providers"]
    if not allowed:
        return True
    name = _norm_provider(provider)
    return name is not None and name in {_norm_provider(a) for a in allowed}


def current_allowlist():
    """Sorted normalized union of every allowlist in MODELS; the probe result records it and the run cell compares."""
    return sorted({_norm_provider(a) for m in MODELS.values() for a in (m["providers"] or [])} - {None})

# Confirmed listed prices, USD per million tokens (input, output), retrieved 2026-11-05 from
# https://openrouter.ai/api/v1/models/<slug>/endpoints for each slug below. Each figure is the highest price
# across that model's OpenRouter endpoints (opus-5: the anthropic/fast priority tier; nemotron: Venice fp8), so
# reservations are worst case. A model whose price is not confirmed must have "in" and "out" set to None: it is
# then left out of PRICE and the notebook refuses to run.
MODELS = {
    "opus-5-low": {
        "slug": "anthropic/claude-opus-5", "effort": "low",
        "in": 10.00, "out": 50.00, "providers": None,
    },
    "nemotron-3-ultra": {
        "slug": "nvidia/nemotron-3-ultra-550b-a55b", "effort": None,
        "in": 0.625, "out": 3.125,        # highest endpoint price, not the DeepInfra rate
        "providers": SAFE_PROVIDERS,
    },
}

MAX_TOKENS = 32000
TIMEOUT_S  = 900
# --- Spend guard -----------------------------------------------------------
# Every OpenRouter call is reserved at its worst case before it is sent and settled after (guarded_post).
# A call is refused, and the run stops, if its model has no entry in PRICE, if its body has no max_tokens,
# or if its worst case would take the run past RUN_BUDGET_USD. PRICE holds each model's confirmed listed price in
# USD per million tokens (input, output). A model with no confirmed price is left out of PRICE and the run refuses to start.
import math
import os
from datetime import datetime, timezone


# --- Canonical env parsing and ledger-root rule (identical in every notebook) ---------------------------------
# RUN_BUDGET_USD: unset means the default. Any value that is set, a blank or whitespace
# value included, must parse as a finite float greater than 0, or the notebook refuses to start.
def _env_usd(name, default):
    raw = os.environ.get(name)
    if raw is None:
        return float(default)
    try:
        v = float(raw.strip())
    except ValueError:
        v = float("nan")
    if not math.isfinite(v) or v <= 0:
        raise RuntimeError(f"Refusing to start: {name} must be a finite number greater than 0 (got {raw!r}).")
    return v


RUN_BUDGET_USD = _env_usd("RUN_BUDGET_USD", 25.00)   # one cap per run; set the RUN_BUDGET_USD environment variable to change it
RUN_ID = None              # None = new run with a fresh ledger. Set a non-empty string to resume and keep its spend.
PRICE = {m["slug"]: (m["in"], m["out"]) for m in MODELS.values()
         if m["in"] is not None and m["out"] is not None}  # confirmed rates in USD per million tokens (input, output)
missing_prices = sorted(set([m["slug"] for m in MODELS.values()]) - set(PRICE))
if missing_prices:
    raise RuntimeError(f"Not run: no confirmed price for {missing_prices}. Look up the listed input and output price "
                       "in USD per million tokens and set 'in' and 'out' for that model in MODELS in this cell, then rerun.")
for _slug, (_pin, _pout) in PRICE.items():
    if not all(isinstance(x, (int, float)) and not isinstance(x, bool) and math.isfinite(x) and x > 0 for x in (_pin, _pout)):
        raise RuntimeError(f"Not run: the price for {_slug} is not a positive finite number. Fix MODELS in this cell, then rerun.")


# Ledger root. If SPEND_LEDGER_DIR is set at all, it is stripped: blank or whitespace-only refuses, anything else
# becomes os.path.abspath(os.path.expanduser(value)). If it is not set, the root is <repo root>/spend_ledger, where
# the repo root is the first directory, starting at the absolute working directory and walking up, for which
# os.path.exists(<dir>/.git) is true (a .git directory or a .git file). With no such directory the notebook refuses:
# there is no fallback to the working directory, because a different working directory would give a different lock.
# The lock is <root>/runner.lock and the run ledgers are <root>/<RUN_ID>/spend_ledger.jsonl.
def _ledger_root():
    raw = os.environ.get("SPEND_LEDGER_DIR")
    if raw is not None:
        value = raw.strip()
        if not value:
            raise RuntimeError("Refusing to start: SPEND_LEDGER_DIR is set but blank. Unset it to use "
                               "<repo root>/spend_ledger, or set it to a directory.")
        return os.path.abspath(os.path.expanduser(value))
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, ".git")):
            return os.path.join(d, "spend_ledger")
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError(f"Refusing to start: SPEND_LEDGER_DIR is not set and there is no .git above the working "
                               f"directory ({start}). Set SPEND_LEDGER_DIR to the shared ledger directory.")
        d = parent


LEDGER_ROOT = _ledger_root()
# None is the only signal for a new run. Any other value is a resume ID and goes through _checked_run_id below,
# which refuses an empty string, 0, False and every other non-string value. Nothing is defaulted.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
def _checked_run_id(value):
    """RUN_ID names one directory under SPEND_ROOT. Refuse an absolute path or any '..' segment, so the text of
    RUN_ID cannot lead os.path.join(SPEND_ROOT, RUN_ID) outside SPEND_ROOT. This is a string check only: it does
    not detect a symlink that already exists under SPEND_ROOT. Fails loudly."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Refusing to start: RUN_ID must be a non-empty string (got {value!r}).")
    if os.path.isabs(value) or value.startswith(("/", "\\")) or os.path.splitdrive(value)[0]:
        raise RuntimeError(f"Refusing to start: RUN_ID must be a relative name, not an absolute path (got {value!r}).")
    if ".." in value.replace("\\", "/").split("/"):
        raise RuntimeError(f"Refusing to start: RUN_ID must not contain a '..' segment (got {value!r}).")
    return value
RUN_ID = _checked_run_id(RUN_ID)
RUN_DIR = os.path.join(LEDGER_ROOT, RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)
LEDGER_PATH = os.path.join(RUN_DIR, "spend_ledger.jsonl")

# One paid runner at a time across all notebooks. An exclusive lock file at LEDGER_ROOT/runner.lock, shared by every
# notebook: a second runner refuses to start while it exists. It is removed on a clean exit only. A stale lock (a
# runner that died without exiting) is never cleared automatically: confirm no runner is alive, then delete
# LEDGER_ROOT/runner.lock by hand to resume.
import atexit
RUNNER_LOCK = os.path.join(LEDGER_ROOT, "runner.lock")
try:
    _lock_fd = os.open(RUNNER_LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
except FileExistsError:
    raise RuntimeError(f"{RUNNER_LOCK} exists. Another runner (any notebook) may be running, or a runner stopped without "
                       "releasing it. Refusing to start. Check for a live runner before deleting the lock by hand.")
_RUNNER_TOKEN = f"pid {os.getpid()} {datetime.now(timezone.utc).isoformat()}"
os.write(_lock_fd, _RUNNER_TOKEN.encode())
os.close(_lock_fd)


def _release_runner_lock():
    try:
        with open(RUNNER_LOCK) as f:
            if f.read() == _RUNNER_TOKEN:
                os.remove(RUNNER_LOCK)
    except OSError:
        pass


atexit.register(_release_runner_lock)


WORKERS    = 8      # concurrent requests; the ledger counts every in-flight reservation, so 8 at once still stay under the cap
RETRIES    = 3

print(f"{len(MODELS)} configs")
print(f"Nemotron pinned to: {', '.join(SAFE_PROVIDERS)} with data_collection=deny")

## 3. Topics

In [ ]:
VOICE = """## Who you are

You are a professional writer with ten years of experience in B2B software. [Placeholder persona: replace with your own.]

This is background so you know whose voice you are writing in. It is not material for the post. Do not list his credentials, employers, degrees, certifications, or biography in the post unless the topic below specifically calls for it. A post that recites this section reads like a resume, not like him.

## Voice rules

Write in lowercase. Do not use sentence case.

No hashtags. No engagement bait; no "thoughts?", no "agree?", no "drop a comment."

Semicolons rather than em-dashes. Use contractions; "i don't" not "i do not."

Sentences under 16 words.

Vary the rhythm. Do not put every sentence on its own line; uniform one-line paragraphs read as machine-generated even when each line is fine.

He does not say "folks," "quick one," "biz," "folks in the space," or "let's dive in." He does not sound like a tech bro.

Do not use: pivotal, crucial, enhance, underscore, landscape, vibrant, testament, foster, delve, showcase, intricate, evolving, cultivate, crafting, utilize, execute as a verb.

No contrast structures. No "it's not X, it's Y." No "not A. Not B. But C." No "despite this."

No cringe transition questions. No "The catch?" No "The kicker?" No "Sound familiar?"

No invented people, anecdotes, dates, dollar figures, or durations. Every specific fact comes from the material below. If you want an example, use one that is already there.

No importance sentences. Nothing about what this means for the broader industry.

No summary ending. No closing line that restates the post.

Do not lecture. No "that's the work." No "that part is the job." No imperative wrap-up telling the reader what to go do.

## Shape

Hook, then receipts, then principle. That order.

The hook is a real observation, not a question and not a statistic dressed as a hook. No "Here's what I learned."

The principle lands at the end and does not preach. Write it in your own words.

Length: 150 to 250 words.

## Output

The post only. No caption, no alternate versions, no explanation."""


TOPICS = {
"T1-benchmark": """Write a LinkedIn post as the persona described below.

## What to write about

He ran a benchmark on which AI model to use for regulatory work. <your benchmark: how many calls, across how many models>. <your headline finding>. <two or three supporting numbers from your benchmark, each with its unit>.

<your own cost comparison; or delete this paragraph>

His conclusion: <your conclusion, in one or two sentences>. A human still reads everything before it ships.

""" + VOICE,

"T2-recall": """Write a LinkedIn post as the persona described below.

## What to write about

<a public recall you can cite: the product, the undeclared allergen, and the supplier change behind it>

<the recall class and scope, as the recall notice states them>

The author's point: supplier approval is not a document you file once. Change notification is the control that fails quietly, because nothing looks wrong until the label is already printed. Most FSQA teams audit the supplier and never audit the supplier's change process.

""" + VOICE,

"T3-client": """Write a LinkedIn post as the persona described below.

## What to write about

<a client story of your own that you can verify: what the client asked for, and where they stood before>

<what your organization delivered, and what the client did next>

The point: <the lesson, in one sentence>.

No client name, no dollar figures, and no invented detail beyond what is stated here.

""" + VOICE,

"T4-farm": """Write a LinkedIn post as the persona described below.

## What to write about

[Placeholder background: replace with your own observation from outside the office.]

The observation: you can tell an animal is sick before any test confirms it. Feed intake drops, the animal separates from the group, posture changes. By the time you have a diagnostic result you already lost days.

The parallel to FSQA: environmental monitoring trends, minor deviations, and near-misses are the feed-intake signal. Most plants wait for the positive result, which is the diagnostic, and then act. The information was available earlier.

""" + VOICE,

"T5-career": """Write a LinkedIn post as the persona described below.

## What to write about

[Placeholder background: replace with your own career story.]

No inspirational arc. No "everything happens for a reason."

""" + VOICE,

"T6-contrarian": """Write a LinkedIn post as the persona described below.

## What to write about

A practice he thinks the food safety industry gets wrong: treating the annual third-party audit as the measure of a food safety program.

His argument: audits are a snapshot on a day the plant knew was coming. A high score tells you the plant can prepare for an audit. It does not tell you what happens on a Tuesday in February when the line is behind and the sanitation window gets cut short.

[Placeholder background: replace with your own experience of this practice.]

He is not saying audits are worthless. He is saying the score is a weak proxy for the thing people use it to measure.

""" + VOICE,

"T7-explainer": """Write a LinkedIn post as the persona described below.

## What to write about

Explaining what an FSVP (Foreign Supplier Verification Program) actually requires, for people who keep confusing it with a certificate.

The substance: FSVP makes the U.S. importer responsible for verifying that a foreign supplier produces food meeting U.S. safety standards. It is not a document a supplier hands you. The importer has to identify hazards, evaluate the supplier's performance, decide on a verification activity, and keep records showing they did it.

The common failure: importers who think a supplier's third-party certificate satisfies FSVP. It can be part of the evaluation. It is not the program.

""" + VOICE,

"T8-announcement": """Write a LinkedIn post as the persona described below.

## What to write about

<your product> is available on <store>. It is <one line on what it does>.

The honest framing: <your own usage facts, and what surprised you about who uses it>. He is treating that as information about who the product is actually for.

No launch hype. No "excited to announce." No metrics beyond what is stated here.

""" + VOICE,
}
print(f"{len(TOPICS)} short-form topics")

## 4. Long-form topics

In [ ]:
FR_DOCS = {
    "fsis_salmonella_initiative_2026-14216.txt":
        "https://www.federalregister.gov/documents/full_text/text/2026/07/15/2026-14216.txt",
    "fda_citrus_red_2_revocation_2026-14909.txt":
        "https://www.federalregister.gov/documents/full_text/text/2026/07/23/2026-14909.txt",
}
def clean(t):
    t = re.sub(r"(?s)<head>.*?</head>", "", t); t = re.sub(r"<[^>]+>", "", t)
    return re.sub(r"\n{3,}", "\n\n", html.unescape(t)).strip()
for name, url in FR_DOCS.items():
    p = SOURCES / name
    if not p.exists():
        p.write_text(clean(httpx.get(url, timeout=90).text), encoding="utf-8")

SOURCE_TEXT = "\n\n---\n\n".join(
    f"[SOURCE FILE: {p.name}]\n\n{p.read_text(encoding='utf-8')}"
    for p in sorted(SOURCES.glob("*.txt")))

REPORT_RULES = """## Citation

Every factual claim traces to the source material below. Cite inline by document name and section or paragraph.

If the source does not answer a question the document needs answered, write that the source does not address it. Do not fill the gap with what is usually true.

Do not quote the source unless the exact wording matters; when you do quote, quote it correctly.

## Voice

Plain and direct. Sentences under 16 words where the meaning allows it.

Do not use: pivotal, crucial, enhance, underscore, landscape, vibrant, testament, foster, delve, showcase, intricate, evolving, cultivate, crafting, utilize, execute as a verb.

No contrast structures. No sentence that tells the reader something is important. No summary ending. No vague authority; name the source or cut the claim.

Semicolons rather than em-dashes. Any statistic gets a date and a named source.

## Output

The document only. No cover note, no explanation of your approach, no offer to revise."""

LONGFORM = {
"L1-report": """You are writing a regulatory intelligence report for a client of a regulatory consultancy. The client is an industry association. The report goes out under the analyst's name and gets read by the client's executive director and passed to its member companies.

Write the report from the source material below. Do not use anything outside it.

## What the report has to do

Explain what happened, what it requires of member companies, and what to watch next. The reader knows food safety; they do not know this specific document. They need to finish reading and know what to do on Monday.

Length: 2,000 words or more.

## Structure

Open with what happened and why the reader is getting this. No preamble about the importance of food safety.

Then the body, organized however the material actually organizes itself.

Close with what to watch next. Specific things: dates, comment periods, follow-on actions, named agencies.

""" + REPORT_RULES + "\n\n---\n\n# SOURCE MATERIAL\n\n" + SOURCE_TEXT,

"L2-memo": """You are writing a one-page action memo for a client of a regulatory consultancy. The client is an industry association. The client's executive director will read this on a phone between meetings and forward it to its members.

Write the memo from the source material below. Do not use anything outside it.

## What the memo has to do

Tell the reader what changed, what they have to do, and by when. Nothing else. A member company should be able to act from this without opening the underlying documents.

Length: 400 to 600 words. Do not exceed 600.

## Structure

Lead with the single thing that requires action, and its deadline.

Then the other items in order of how soon they matter.

Anything that does not require action from this client gets one sentence saying so, or gets cut. Judge which items those are; do not give every item equal weight.

""" + REPORT_RULES + "\n\n---\n\n# SOURCE MATERIAL\n\n" + SOURCE_TEXT,
}

PROMPTS = {**TOPICS, **LONGFORM}
TOTAL = len(PROMPTS) * len(MODELS)
print(f"{len(PROMPTS)} prompts x {len(MODELS)} configs = {TOTAL} calls")

## 5. Probe

Two calls. The Nemotron probe prints the provider that actually served it. If that comes
back as anything other than DeepInfra or Baseten, stop and fix the routing before running.

In [ ]:
import threading, uuid
from decimal import Decimal


class BudgetExceeded(RuntimeError):
    pass


PROMPT_OVERHEAD_TOKENS = 256    # allowance for chat-template tokens that the byte bound below does not cover


def max_price_for(model):
    """provider.max_price for one model: its listed price in USD per million tokens (input, output). PRICE already
    holds per-million figures, so the value is converted with Decimal and not scaled. An unpriced model is refused."""
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    pin, pout = PRICE[model]
    return {"prompt": float(Decimal(str(pin))), "completion": float(Decimal(str(pout)))}


def openrouter_body(body):
    """The body every OpenRouter call sends: the caller's body with provider.max_price set from PRICE and usage
    accounting switched on. Done here, once, so no call site can leave either out."""
    b = dict(body)
    b["provider"] = {**(body.get("provider") or {}), "max_price": max_price_for(body["model"])}
    b["usage"] = {"include": True}
    return b


def worst_case_cost(body):
    """Upper bound for one attempt. Input comes from the UTF-8 byte count of the messages (each token covers
    at least one byte); output is capped by max_tokens, which includes reasoning. Fails closed."""
    model = body["model"]
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    if not isinstance(body.get("max_tokens"), int):
        raise BudgetExceeded(f"not sent: request for {model} has no max_tokens, so its cost cannot be bounded.")
    pin, pout = PRICE[model]
    in_bound = len(json.dumps(body["messages"], ensure_ascii=False).encode("utf-8")) + PROMPT_OVERHEAD_TOKENS
    return pin / 1e6 * in_bound + pout / 1e6 * body["max_tokens"]


def _is_count(x):
    return isinstance(x, int) and not isinstance(x, bool) and x >= 0


def attempt_cost(body, usage):
    """Settle only against complete, validated provider usage: a reported cost that is finite and above zero,
    and both token counts as non-negative integers. Anything less keeps the worst case that was reserved, so a
    zero, missing, partial or malformed reply cannot release the reservation."""
    if isinstance(usage, dict):
        c = usage.get("cost")
        if (isinstance(c, (int, float)) and not isinstance(c, bool) and 0 < c < float("inf")
                and _is_count(usage.get("prompt_tokens")) and _is_count(usage.get("completion_tokens"))):
            return float(c), "usage"
    return worst_case_cost(body), "worst_case"


_ledger_lock = threading.Lock()
_ledger = {}                    # attempt id -> entry; reloaded from LEDGER_PATH on resume
_budget_halted = False          # set by a refused dispatch or by an overrun; no later attempt is sent
if os.path.exists(LEDGER_PATH):
    with open(LEDGER_PATH) as _f:
        for _line in _f:
            if _line.strip():
                _rec = json.loads(_line)
                _ledger.setdefault(_rec["id"], {}).update(_rec)
    _budget_halted = any(_e.get("overrun") for _e in _ledger.values())   # an overrun halts the run after a restart too


def ledger_total():             # caller holds _ledger_lock. A reserved attempt with no settled line counts at worst case.
    return sum(e["cost"] if e["state"] == "settled" else e["worst_case"] for e in _ledger.values())


def _ledger_write(rec):
    with open(LEDGER_PATH, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def ledger_reserve(body, purpose):
    """Reserve one attempt at its worst case before it is sent. Raises BudgetExceeded if the run is halted, or if
    the reservation would take the run past RUN_BUDGET_USD; landing exactly on the cap is allowed."""
    global _budget_halted
    wc = round(worst_case_cost(body), 6)
    with _ledger_lock:
        if _budget_halted:
            raise BudgetExceeded(f"not sent: the run is halted, so {body['model']} ({purpose}) is refused before dispatch")
        if ledger_total() + wc <= RUN_BUDGET_USD:
            aid = uuid.uuid4().hex
            e = {"id": aid, "state": "reserved", "model": body["model"], "purpose": purpose,
                 "worst_case": wc, "ts": datetime.now(timezone.utc).isoformat()}
            _ledger[aid] = e
            _ledger_write(e)
            return aid
        _budget_halted = True
    raise BudgetExceeded(f"not sent: {body['model']} ({purpose}), worst case ${wc:.4f}, "
                         f"would take the run past RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}")


def ledger_settle(aid, body, usage):
    """Settle one attempt. A reported charge above the reservation for this attempt is an overrun: it is written to
    the ledger with its reservation and halts the run, so every later dispatch is refused before any request is sent."""
    global _budget_halted
    cost, source = attempt_cost(body, usage)
    with _ledger_lock:
        e = _ledger[aid]
        charged = round(cost, 6)
        overrun = source == "usage" and charged > e["worst_case"]
        e.update(state="settled", cost=charged, cost_source=source)
        rec = {"id": aid, "state": "settled", "cost": charged, "cost_source": source}
        if overrun:
            e["overrun"] = True
            rec.update(overrun=True, reserved=e["worst_case"])
            _budget_halted = True
        _ledger_write(rec)
    if overrun:
        print(f"Ledger overrun: an attempt was charged ${charged:.4f} against a reservation of ${e['worst_case']:.4f}. "
              f"The run is halted; no later attempt will be sent.")
    return e["cost"]


def ledger_summary():
    with _ledger_lock:
        es, total = list(_ledger.values()), ledger_total()
    by = {}
    for e in es:
        n, c = by.get(e["purpose"], (0, 0.0))
        by[e["purpose"]] = (n + 1, c + (e["cost"] if e["state"] == "settled" else e["worst_case"]))
    lines = [f"Spend ledger: {len(es)} attempts, RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}"]
    lines += [f"  {p:8s} attempts={n:5d}  charged=${c:.4f}" for p, (n, c) in sorted(by.items())]
    lines.append(f"  total charged ${total:.4f}   reserved, not settled={sum(e['state'] == 'reserved' for e in es)}")
    return "\n".join(lines)


def guarded_post(body, headers, timeout=300, purpose="call"):
    """One POST to OpenRouter. The attempt is reserved at its worst case before it is sent and settled after, so
    retries, exceptions and rejected calls are all counted. BudgetExceeded is raised before anything is sent."""
    body = openrouter_body(body)
    aid = ledger_reserve(body, purpose)
    usage = None
    try:
        r = httpx.post(URL, headers=headers, json=body, timeout=timeout)
        if r.status_code == 200:
            try:
                usage = (r.json() or {}).get("usage") or None
            except Exception:
                usage = None
        return r
    finally:
        ledger_settle(aid, body, usage)


URL = "https://openrouter.ai/api/v1/chat/completions"

def headers():
    return {"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
            "Content-Type": "application/json",
            "HTTP-Referer": "https://example.com/labs-benchmark",
            "X-Title": "labs-benchmark"}

def build(cfg, messages, max_tokens):
    m = MODELS[cfg]
    p = {"model": m["slug"], "messages": messages, "max_tokens": max_tokens,
         "temperature": 1.0, "usage": {"include": True}}
    if m["effort"]:
        p["reasoning"] = {"effort": m["effort"]}
    if m["providers"]:
        p["provider"] = {"only": m["providers"],
                         "data_collection": "deny",
                         "allow_fallbacks": False}
    return p

def cost_of(cfg, ptok, ctok):
    m = MODELS[cfg]
    if ptok is None or ctok is None: return None
    return ptok / 1e6 * m["in"] + ctok / 1e6 * m["out"]

def call(cfg, messages, max_tokens=None):
    mt = max_tokens or MAX_TOKENS
    p = build(cfg, messages, mt); last = None
    for i in range(RETRIES):
        t0 = time.time()
        try:
            r = guarded_post(p, headers(), timeout=TIMEOUT_S, purpose=cfg)
            el = round(time.time() - t0, 2)
            if r.status_code in (400, 404):
                return {"ok": False, "error": f"{r.status_code} {r.text[:300]}", "latency_s": el}
            r.raise_for_status()
            d = r.json()
            if "error" in d:
                return {"ok": False, "error": str(d["error"])[:300], "latency_s": el}
            ch = d["choices"][0]; text = ch["message"].get("content"); u = d.get("usage") or {}
            if not text or not text.strip():
                last = f"empty content (finish={ch.get('finish_reason')})"
                if i < RETRIES - 1:
                    mt = int(mt * 1.5); p = build(cfg, messages, mt); continue
                return {"ok": False, "error": last, "latency_s": el}
            return {"ok": True, "text": text, "latency_s": el,
                    "provider": d.get("provider"),
                    "finish_reason": ch.get("finish_reason"),
                    "prompt_tokens": u.get("prompt_tokens"),
                    "completion_tokens": u.get("completion_tokens"),
                    "reasoning_tokens": (u.get("completion_tokens_details") or {}).get("reasoning_tokens"),
                    "cost_reported": u.get("cost"),
                    "cost_computed": cost_of(cfg, u.get("prompt_tokens"), u.get("completion_tokens"))}
        except BudgetExceeded:
            raise
        except Exception as e:
            last = f"{type(e).__name__}: {e}"
            if i < RETRIES - 1: time.sleep(2 ** i)
    return {"ok": False, "error": last or "unknown", "latency_s": None}


# Provider probe. ok_to_run is True only if every probe call succeeded and, for each model that has a provider
# allowlist, the response named an explicitly allowlisted provider. Missing metadata counts as a failure. The result
# is recorded in PROBE_RESULT, bound to the model ids and the allowlist it was taken with; the run cell checks it.
ok_to_run = False
PROBE_RESULT = {"ok": False, "provider": None, "reason": "the provider probe did not finish", "models": {},
                "allowlist": current_allowlist()}
_probed = {}
for name in MODELS:
    entry = _probed[name] = {"model": MODELS[name]["slug"], "ok": False, "provider": None, "reason": ""}
    try:
        res = call(name, [{"role": "user", "content": "Reply with the single word: ok"}], 2000)
    except Exception as e:
        entry["reason"] = f"the probe call for {name} raised {type(e).__name__}"
        print(f"  FAIL  {name:18s} {entry['reason']}")
        continue
    if not isinstance(res, dict) or not res.get("ok"):
        entry["reason"] = f"the probe call for {name} did not succeed"
        print(f"  FAIL  {name:18s} {str((res or {}).get('error') if isinstance(res, dict) else res)[:140]}")
        continue
    prov = res.get("provider")
    print(f"  OK    {name:18s} {res.get('latency_s'):>6}s  provider={prov}  "
          f"rtok={res.get('reasoning_tokens')}")
    entry["provider"] = prov if isinstance(prov, str) else None
    if not MODELS[name]["providers"]:
        entry["ok"] = True
    elif _norm_provider(prov) is None:
        entry["reason"] = f"the probe response for {name} carried no provider name, so its route is unverified"
        print(f"        ROUTING PROBLEM: no provider metadata in the response for {name}.")
        print(f"        Do not run. The route cannot be verified.")
    elif not provider_ok(name, prov):
        entry["reason"] = f"{name} was served by a provider that is not on its allowlist"
        print(f"        ROUTING PROBLEM: served by {prov}, not in {MODELS[name]['providers']}.")
        print(f"        Do not run. Confidential material could reach a logging endpoint.")
    else:
        entry["ok"] = True

_failed = [e for e in _probed.values() if not e["ok"]]
PROBE_RESULT = {"ok": not _failed,
                "provider": next((e["provider"] for n, e in _probed.items() if MODELS[n]["providers"]), None),
                "reason": "; ".join(e["reason"] for e in _failed) if _failed else "ok",
                "models": _probed, "allowlist": current_allowlist()}
ok_to_run = PROBE_RESULT["ok"]

print("\nSafe to run." if ok_to_run else "\nFix the failures above before running.")

## 6. Run

In [ ]:
# Provider gate. Runs before anything is reserved or built. The run proceeds only on a probe result that succeeded,
# named an allowlisted provider (missing metadata is a failure), and was taken for these same model ids and this
# same allowlist. A flag left over from an earlier probe, or set by hand, is not enough.
def _probe_refusal():
    try:
        flag, pr = ok_to_run, PROBE_RESULT
    except NameError:
        return "the provider probe has not been run in this session; run the probe cell first"
    if flag is not True or not isinstance(pr, dict) or pr.get("ok") is not True:
        why = pr.get("reason") if isinstance(pr, dict) else None
        return str(why) if why else "the provider probe did not succeed"
    probed = pr.get("models")
    if not isinstance(probed, dict) or set(probed) != set(MODELS):
        return "the probe result does not cover the models this run uses; rerun the probe cell"
    for cfg, m in MODELS.items():
        e = probed[cfg]
        if not isinstance(e, dict) or e.get("model") != m["slug"] or e.get("ok") is not True:
            return f"the probe result was not taken for the current model of {cfg}; rerun the probe cell"
        if not provider_ok(cfg, e.get("provider")):
            return f"the probe result for {cfg} does not show an allowlisted provider; rerun the probe cell"
    if pr.get("allowlist") != current_allowlist():
        return "the provider allowlist changed after the probe; rerun the probe cell"
    return None


_why = _probe_refusal()
if _why:
    raise RuntimeError("Refusing to run: " + _why)

done = set()
if RAW.exists():
    for line in RAW.read_text(encoding="utf-8").splitlines():
        if line.strip():
            r = json.loads(line)
            if r.get("ok") and isinstance(r.get("text"), str) and r["text"].strip():
                done.add((r["model"], r["topic"]))

jobs = [(m, t) for t in PROMPTS for m in MODELS if (m, t) not in done]
print(f"{len(jobs)} to run, {len(done)} done.\n")

def work(j):
    m, t = j
    res = call(m, [{"role": "user", "content": PROMPTS[t]}])
    res.update({"model": m, "topic": t, "kind": "long" if t.startswith("L") else "short",
                "slug": MODELS[m]["slug"], "effort": MODELS[m]["effort"],
                "ts": datetime.now(timezone.utc).isoformat()})
    return res

if jobs:
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futs = [pool.submit(work, j) for j in jobs]
        with RAW.open("a", encoding="utf-8") as fh:
            for f in as_completed(futs):
                r = f.result()
                fh.write(json.dumps(r, ensure_ascii=False) + "\n"); fh.flush()
                if r["ok"]:
                    print(f"  done  {r['topic']:18s} {r['model']:18s} "
                          f"{len(r['text'].split()):>5}w  {r['latency_s']:>6}s  "
                          f"${r.get('cost_computed') or 0:.4f}  {r.get('provider')}")
                else:
                    print(f"  FAIL  {r['topic']:18s} {r['model']:18s} {r['error'][:60]}")

# Post-run audit: every Nemotron call must have been served by an allowed provider.
served = {}
for line in RAW.read_text(encoding="utf-8").splitlines():
    if line.strip():
        r = json.loads(line)
        if r.get("ok") and r["model"] == "nemotron-3-ultra":
            served[str(r.get("provider"))] = served.get(str(r.get("provider")), 0) + 1
if served:
    print("\nNemotron served by:", served)
    bad = [p for p in served if not provider_ok("nemotron-3-ultra", p)]
    print("ROUTING VIOLATION:", bad) if bad else print("All Nemotron calls stayed on allowed providers.")

## 7. Reading pack

Two outputs per topic, A / B, reshuffled each time. Rank first and second only.

In [ ]:
recs = []
for line in RAW.read_text(encoding="utf-8").splitlines():
    if line.strip():
        r = json.loads(line)
        if r.get("ok") and isinstance(r.get("text"), str) and r["text"].strip():
            recs.append(r)
latest = {(r["model"], r["topic"]): r for r in recs}
recs = list(latest.values())

by_topic = {}
for r in recs:
    by_topic.setdefault(r["topic"], []).append(r)

pack = ["# Round 6 reading pack — Nemotron 3 Ultra vs Opus 5 low", "",
        f"Generated {datetime.now(timezone.utc):%Y-%m-%d %H:%M} UTC. {len(recs)} of {TOTAL}.", "",
        "Two per topic, A / B, reshuffled every topic. Pick a winner for each.", ""]
key = ["# Answer key — round 6", "", "Do not read before ranking.", ""]

for topic in sorted(by_topic):
    rs = by_topic[topic][:]; random.shuffle(rs)
    kind = "long-form" if topic.startswith("L") else "short-form"
    pack += [f"## {topic}  ({kind})", ""]
    letters = "AB"[:len(rs)]
    for letter, r in zip(letters, rs):
        pack += [f"### {topic} — {letter}  ({len(r['text'].split())} words)", "",
                 r["text"].strip(), "", "---", ""]
    key.append(f"**{topic}** — " + ", ".join(
        f"{l} = {r['model']}" for l, r in zip(letters, rs)))

key += ["", "## Metrics", "",
        "| topic | config | words | latency | $ | rtok | provider |",
        "|---|---|---|---|---|---|---|"]
for r in sorted(recs, key=lambda x: (x["topic"], x["model"])):
    key.append(f"| {r['topic']} | {r['model']} | {len(r['text'].split())} | {r['latency_s']}s | "
               f"${r.get('cost_computed') or 0:.4f} | {r.get('reasoning_tokens')} | "
               f"{r.get('provider')} |")

READING.write_text("\n".join(pack), encoding="utf-8")
ANSWERS.write_text("\n".join(key), encoding="utf-8")
if not RANKS.exists():
    RANKS.write_text("topic,winner,notes\n" + "\n".join(f"{t},," for t in sorted(by_topic)),
                     encoding="utf-8")

print(f"pack -> {READING} ({READING.stat().st_size/1024:.0f} KB)")
print(f"key  -> {ANSWERS}  (do not open yet)")
print(f"ranks-> {RANKS}    fill with:  T1-benchmark,A,")

## 8. Tally

In [ ]:
import csv
mapping = {}
for line in ANSWERS.read_text(encoding="utf-8").splitlines():
    if line.startswith("**"):
        t = line.split("**")[1]
        mapping[t] = dict(p.strip().split(" = ") for p in line.split("—", 1)[1].split(","))

rows = [r for r in csv.DictReader(RANKS.open(encoding="utf-8")) if (r.get("winner") or "").strip()]
if not rows:
    raise SystemExit("rankings.csv is empty.")

wins = {m: 0 for m in MODELS}
for r in rows:
    m = mapping.get(r["topic"].strip(), {}).get((r["winner"] or "").strip().upper())
    if m: wins[m] += 1
    print(f"{r['topic']:18s} {m}")

n = len(rows)
print(f"\n{'config':20s} {'wins':>5s}")
print("-" * 27)
for m, w in sorted(wins.items(), key=lambda kv: -kv[1]):
    print(f"{m:20s} {w:>5d} / {n}")

top = max(wins, key=wins.get)
print(f"\n{top} won {wins[top]} of {n}. Coin-flip baseline is {n/2:.0f}.")
print("8 of 10 is p=0.055 two-tailed; 9 of 10 is p=0.021. A 6-4 split is noise,")
print("and a genuine tie makes Nemotron the answer on cost alone at ~1/10th the rate.")